# Redes Neurais - Projeto 2: Quantização de cores com SOM e GNG
### Julia Souza Belmonte Lopes - RA: 168906

A ideia do projeto é reduzir a paleta de uma imagem usando aprendizado competitivo. Cada pixel é uma amostra com 3 atributos (R, G, B) e cada neurônio da rede treinada vira uma cor da nova paleta. Treinamos SOM com grades 4x4, 8x8 e 16x16 e GNG com 16, 64 e 256 neurônios, avaliamos com o erro de quantização e o erro topológico e reconstruímos a imagem trocando cada pixel pela cor do neurônio vencedor.

Os dois algoritmos foram implementados aqui mesmo, só com NumPy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42

## 1. Imagem

A imagem é reduzida para no máximo 256 pixels de lado para o treino não demorar. Depois disso todos os pixels são usados, com os valores RGB normalizados para [0, 1].

In [ ]:
CAMINHO = "imagem.jpg"

img = Image.open(CAMINHO).convert("RGB")
img.thumbnail((256, 256))
img = np.asarray(img)
altura, largura, _ = img.shape

X = img.reshape(-1, 3) / 255.0   # uma linha por pixel
cores_original = len(np.unique(img.reshape(-1, 3), axis=0))

print("Tamanho:", largura, "x", altura)
print("Pixels:", len(X))
print("Cores distintas:", cores_original)

plt.imshow(img)
plt.title("Imagem original")
plt.axis("off")
plt.show()

## 2. Métricas

O erro de quantização (QE) é a distância média entre cada pixel e o seu neurônio vencedor (BMU), no espaço RGB normalizado. Quanto menor, mais parecidas ficam as cores.

O erro topológico (TE) é a fração de pixels em que o 1º e o 2º neurônios mais próximos não são vizinhos na rede. No SOM, vizinho é quem está ao lado na grade (contando as diagonais). No GNG, é quem está ligado por uma aresta.

In [ ]:
def dois_bmus(X, W):
    # 1º e 2º neurônios mais próximos de cada pixel e a distância até o 1º
    b1, b2, d1 = [], [], []
    for i in range(0, len(X), 4096):   # em blocos, para não usar muita memória
        d = np.linalg.norm(X[i:i + 4096, None, :] - W[None, :, :], axis=2)
        ordem = np.argsort(d, axis=1)
        b1.append(ordem[:, 0])
        b2.append(ordem[:, 1])
        d1.append(d.min(axis=1))
    return np.concatenate(b1), np.concatenate(b2), np.concatenate(d1)


def avaliar(X, W, vizinhos):
    # vizinhos[i, j] = True se os neurônios i e j são vizinhos
    b1, b2, d1 = dois_bmus(X, W)
    qe = d1.mean()
    te = np.mean(~vizinhos[b1, b2])
    return qe, te


def reconstruir(X, W):
    b1, _, _ = dois_bmus(X, W)
    return W[b1].reshape(altura, largura, 3)

## 3. SOM

Os neurônios ficam em uma grade fixa. A cada passo sorteamos um pixel, achamos o BMU e puxamos o BMU e os vizinhos dele na grade em direção ao pixel:

$$w_i \leftarrow w_i + \eta \, h_{ci} \, (x - w_i)$$

onde $h_{ci}$ é uma gaussiana da distância na grade entre o neurônio $i$ e o BMU $c$. A taxa $\eta$ e o raio da vizinhança diminuem exponencialmente durante o treino.

In [ ]:
class SOM:
    def __init__(self, lado, taxa_ini, taxa_fim, seed=SEED):
        self.lado = lado
        self.taxa_ini, self.taxa_fim = taxa_ini, taxa_fim
        self.raio_ini, self.raio_fim = lado / 2, 0.5
        self.rng = np.random.default_rng(seed)
        # posição (linha, coluna) de cada neurônio na grade
        self.pos = np.array([(i, j) for i in range(lado) for j in range(lado)])

    def treinar(self, X, passos):
        # pesos iniciais: pixels sorteados da própria imagem
        self.W = X[self.rng.choice(len(X), self.lado ** 2, replace=False)].copy()
        for t in range(passos):
            f = t / passos
            taxa = self.taxa_ini * (self.taxa_fim / self.taxa_ini) ** f
            raio = self.raio_ini * (self.raio_fim / self.raio_ini) ** f

            x = X[self.rng.integers(len(X))]
            bmu = np.argmin(((self.W - x) ** 2).sum(axis=1))
            d_grade = ((self.pos - self.pos[bmu]) ** 2).sum(axis=1)
            h = np.exp(-d_grade / (2 * raio ** 2))
            self.W += taxa * h[:, None] * (x - self.W)
        return self

    def vizinhos(self):
        d = np.abs(self.pos[:, None, :] - self.pos[None, :, :]).max(axis=2)
        return d == 1

## 4. GNG

O GNG começa com 2 neurônios e vai crescendo. A cada pixel sorteado:

1. acha os dois neurônios mais próximos (s1 e s2);
2. envelhece as arestas de s1 e soma o erro ao acumulado de s1;
3. move s1 (taxa `eps_b`) e os vizinhos dele (taxa `eps_n`) na direção do pixel;
4. cria ou renova a aresta s1-s2, apaga as arestas velhas demais e os neurônios que ficaram sem aresta;
5. a cada `lam` passos insere um neurônio entre o de maior erro e o vizinho dele de maior erro, até chegar em `n_max`.

Depois que a rede atinge `n_max` o treino continua, só ajustando os pesos e as arestas.

In [ ]:
class GNG:
    def __init__(self, n_max, eps_b, eps_n, lam=100, idade_max=50,
                 alpha=0.5, d=0.995, seed=SEED):
        self.n_max = n_max
        self.eps_b, self.eps_n = eps_b, eps_n
        self.lam, self.idade_max = lam, idade_max
        self.alpha, self.d = alpha, d
        self.rng = np.random.default_rng(seed)

    def treinar(self, X, passos):
        n = 2
        W = np.zeros((self.n_max, 3))
        W[:2] = X[self.rng.choice(len(X), 2, replace=False)]
        erro = np.zeros(self.n_max)
        idade = np.full((self.n_max, self.n_max), -1)   # -1 = sem aresta

        for t in range(1, passos + 1):
            x = X[self.rng.integers(len(X))]
            d2 = ((W[:n] - x) ** 2).sum(axis=1)
            s1, s2 = np.argsort(d2)[:2]

            viz = np.where(idade[s1, :n] >= 0)[0]
            idade[s1, viz] += 1
            idade[viz, s1] += 1
            erro[s1] += d2[s1]
            W[s1] += self.eps_b * (x - W[s1])
            W[viz] += self.eps_n * (x - W[viz])
            idade[s1, s2] = idade[s2, s1] = 0

            # arestas velhas e neurônios isolados
            velhas = np.where(idade[s1, :n] > self.idade_max)[0]
            idade[s1, velhas] = idade[velhas, s1] = -1
            for i in sorted(velhas, reverse=True):
                if (idade[i, :n] >= 0).any():
                    continue
                # o último neurônio ocupa o lugar do removido
                u = n - 1
                W[i], erro[i] = W[u], erro[u]
                idade[i, :], idade[:, i] = idade[u, :].copy(), idade[:, u].copy()
                idade[i, i] = -1
                idade[u, :] = idade[:, u] = -1
                n -= 1

            # inserção
            if t % self.lam == 0 and n < self.n_max:
                q = np.argmax(erro[:n])
                viz = np.where(idade[q, :n] >= 0)[0]
                f = viz[np.argmax(erro[viz])]
                W[n] = (W[q] + W[f]) / 2
                idade[q, f] = idade[f, q] = -1
                idade[q, n] = idade[n, q] = 0
                idade[f, n] = idade[n, f] = 0
                erro[q] *= self.alpha
                erro[f] *= self.alpha
                erro[n] = erro[q]
                n += 1

            erro[:n] *= self.d

        self.W = W[:n]
        self.arestas = idade[:n, :n] >= 0
        return self

    def vizinhos(self):
        return self.arestas

## 5. Treinamento e ajuste das taxas

Todas as redes recebem o mesmo número de atualizações, para a comparação entre SOM e GNG ser justa. Para cada tamanho testamos três combinações de taxas de aprendizagem:

- SOM: taxa inicial e final (a final é sempre um décimo da inicial);
- GNG: taxa do vencedor (`eps_b`) e dos vizinhos (`eps_n`). A primeira combinação é a do artigo do Fritzke.

Os outros parâmetros ficam fixos.

In [ ]:
PASSOS = 40000

taxas_som = [(0.9, 0.09), (0.5, 0.05), (0.1, 0.01)]
taxas_gng = [(0.2, 0.006), (0.05, 0.006), (0.01, 0.001)]

redes = {}
linhas = []

for lado in [4, 8, 16]:
    for taxas in taxas_som:
        som = SOM(lado, *taxas).treinar(X, PASSOS)
        qe, te = avaliar(X, som.W, som.vizinhos())
        nome = f"SOM {lado}x{lado}"
        redes[(nome, taxas)] = som
        linhas.append([nome, len(som.W), taxas, qe, te])

for n in [16, 64, 256]:
    for taxas in taxas_gng:
        gng = GNG(n, *taxas).treinar(X, PASSOS)
        qe, te = avaliar(X, gng.W, gng.vizinhos())
        nome = f"GNG {n}"
        redes[(nome, taxas)] = gng
        linhas.append([nome, len(gng.W), taxas, qe, te])

tabela = pd.DataFrame(linhas, columns=["Rede", "Neurônios", "Taxas", "QE", "TE"])
tabela.round(4)

Para cada rede ficamos com a combinação de menor QE, já que o objetivo aqui é aproximar as cores da imagem.

In [ ]:
melhores = tabela.loc[tabela.groupby("Rede", sort=False)["QE"].idxmin()].reset_index(drop=True)

imagens = {}
cores_usadas = []
for _, linha in melhores.iterrows():
    rede = redes[(linha["Rede"], linha["Taxas"])]
    rec = reconstruir(X, rede.W)
    imagens[linha["Rede"]] = rec
    cores_usadas.append(len(np.unique(rec.reshape(-1, 3), axis=0)))

melhores["Cores usadas"] = cores_usadas
melhores.round(4)

## 6. Imagens reconstruídas

Cada pixel recebe a cor (vetor de pesos) do seu BMU.

In [ ]:
fig, eixos = plt.subplots(2, 4, figsize=(16, 8))
for i, tipo in enumerate(["SOM", "GNG"]):
    eixos[i, 0].imshow(img)
    eixos[i, 0].set_title("Original")
    nomes = [n for n in imagens if n.startswith(tipo)]
    for ax, nome in zip(eixos[i, 1:], nomes):
        qe = melhores.loc[melhores["Rede"] == nome, "QE"].item()
        ax.imshow(imagens[nome])
        ax.set_title(f"{nome} (QE = {qe:.4f})")
for ax in eixos.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Paletas e topologia

Em cima, os pesos de cada SOM mostrados na própria grade: cada quadrado é um neurônio. Embaixo, os neurônios do GNG nos eixos R e G, com as arestas que a rede aprendeu.

In [ ]:
fig, eixos = plt.subplots(2, 3, figsize=(13, 8))

for ax, nome in zip(eixos[0], [n for n in imagens if n.startswith("SOM")]):
    taxas = melhores.loc[melhores["Rede"] == nome, "Taxas"].item()
    som = redes[(nome, taxas)]
    ax.imshow(som.W.reshape(som.lado, som.lado, 3))
    ax.set_title(nome)
    ax.axis("off")

for ax, nome in zip(eixos[1], [n for n in imagens if n.startswith("GNG")]):
    taxas = melhores.loc[melhores["Rede"] == nome, "Taxas"].item()
    gng = redes[(nome, taxas)]
    for i, j in zip(*np.where(np.triu(gng.arestas))):
        ax.plot(gng.W[[i, j], 0], gng.W[[i, j], 1], color="gray", lw=0.6, zorder=1)
    ax.scatter(gng.W[:, 0], gng.W[:, 1], c=gng.W, s=45, edgecolors="black", linewidths=0.4, zorder=2)
    ax.set_title(nome)
    ax.set_xlabel("R")
    ax.set_ylabel("G")

plt.tight_layout()
plt.show()

## 8. Comparação dos erros

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
for tipo in ["SOM", "GNG"]:
    dados = melhores[melhores["Rede"].str.startswith(tipo)]
    eixos[0].plot(dados["Neurônios"], dados["QE"], marker="o", label=tipo)
    eixos[1].plot(dados["Neurônios"], dados["TE"], marker="o", label=tipo)

eixos[0].set_ylabel("Erro de quantização")
eixos[1].set_ylabel("Erro topológico")
for ax in eixos:
    ax.set_xscale("log", base=2)
    ax.set_xticks([16, 64, 256])
    ax.set_xticklabels([16, 64, 256])
    ax.set_xlabel("Neurônios")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 9. Conclusão

Com 16 cores a imagem continua reconhecível, mas os degradês viram faixas e os detalhes pequenos perdem a cor. Com 64 a diferença para a original já é pequena e com 256 quase não dá para perceber, apesar de a original ter muito mais cores (valor impresso na seção 1). Em termos de compressão, cada pixel deixa de ocupar 24 bits e passa a ser um índice de 4, 6 ou 8 bits, mais a paleta.

O QE confirma o que se vê nas imagens: ele cai sempre que o número de neurônios aumenta, nos dois algoritmos. Para o mesmo número de neurônios o GNG teve QE um pouco menor que o SOM, e essa diferença diminui nas redes maiores. No SOM a vizinhança da grade faz os neurônios vizinhos se moverem juntos, o que limita o ajuste de cada um. Já o GNG insere os neurônios novos onde o erro acumulado é maior.

No erro topológico a diferença é bem maior: o TE do GNG fica perto de zero e o do SOM é várias vezes maior. Isso vem de como cada rede define vizinhança. O GNG cria as arestas ligando o 1º e o 2º vencedores, então a topologia dele se adapta aos dados. O SOM precisa encaixar as cores, que estão em 3 dimensões, em uma grade fixa de 2 dimensões, e nem sempre cores próximas conseguem ficar lado a lado. Mesmo assim, as grades da seção 7 mostram que o SOM organizou as cores, com tons parecidos em regiões vizinhas do mapa.

Sobre as taxas, a diferença entre as combinações testadas foi pequena perto da diferença causada pelo número de neurônios. As taxas mais baixas tenderam a dar QE maior, principalmente no GNG de 256 neurônios. A combinação de menor QE nem sempre foi a de menor TE, então a escolha depende do que se quer priorizar. Aqui priorizamos o QE.

Vale lembrar que os resultados são de uma única imagem e de uma única semente.

## Referências

- Aula 12B - Projeto 2: Aprendizado Não Supervisionado (material da disciplina).
- KOHONEN, T. The self-organizing map. *Proceedings of the IEEE*, v. 78, n. 9, p. 1464-1480, 1990.
- FRITZKE, B. A growing neural gas network learns topologies. *Advances in Neural Information Processing Systems 7*, p. 625-632, 1995.